In [7]:
import numpy as np
from scipy.stats import pearsonr
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Carguemos la matriz usuarios-ítems: 
ui_matrix = load_npz('../data/processed/ui_matrix_csr.npz')

### 1) Cálculo de similaridad: 

In [3]:
# Decidimos usar la correlación de Pearson como métrica de similaridad: 

def pearson_similarity_users(ui_matrix, user_a, user_b, min_common=3):
    """
    Correlación de Pearson entre dos usuarios usando solo ítems en común.
    Se establece como parámetro opcional 'min_common', que establece el número mínimo de 
    ítems en común para calcularse su similaridad. 
    """

    # Obtenemos los vectores de cada usuario (a y b):
    va = ui_matrix[user_a].toarray().ravel()
    vb = ui_matrix[user_b].toarray().ravel()

    # Devolvemos 0 en caso de que existan menos ítems en común de los necesarios: 
    common = (va > 0) & (vb > 0)
    if common.sum() < min_common:
        return 0.0

    # Filtramos solo aquellos ítems comunes: 
    va = va[common]
    vb = vb[common]

    # Si alguno no tiene variación, Pearson no está definido:
    if np.std(va) == 0 or np.std(vb) == 0:
        return 0.0

    # Calculamos la correlación de Pearson y devolvemos su valor:
    sim, _ = pearsonr(va, vb)
    if np.isnan(sim):
        return 0.0
    return float(sim)

# Ejemplo de prueba:
user_a = 0
user_b = 100
pearson_similarity_users(ui_matrix, 0, 100)

0.0

In [8]:
# Se observa en el paso 2 que, usando la correlación de Pearson, todos los usuarios 
# con respecto a uno dado parecen tener 0 de similaridad (al menos en las 
# pruebas realizadas). Por ello, probaremos a usar la distancia coseno 
# (usando para ello la función de sklearn): 

def cosine_similarity_users(ui_matrix, user_a, user_b, min_common=1):
    """
    Similaridad coseno entre dos usuarios usando sklearn.
    """
    va = ui_matrix[user_a]
    vb = ui_matrix[user_b]

    # Exigimos un mínimo de ítems en común: 
    common_count = len(set(va.indices) & set(vb.indices))
    if common_count < min_common:
        return 0.0

    sim = cosine_similarity(va, vb)[0, 0]
    if np.isnan(sim):
        return 0.0
    return float(sim)

# Ejemplo de prueba:
user_a = 0
user_b = 100
cosine_similarity_users(ui_matrix, 0, 100)

0.0

### 2) Búsqueda de los *k* vecinos: 

In [9]:
# Seguidamente, definimos la siguiente función para obtener los k usuarios más similares a un usuario objetivo:

def knn_users(ui_matrix, target_user, similarity, k=5, min_common=3):
    """Devuelve los k usuarios más similares al usuario objetivo."""

    n_users = ui_matrix.shape[0]
    sims = []

    for other_user in range(n_users):
        if other_user == target_user:
            continue
        sim = similarity(
            ui_matrix, target_user, other_user, min_common=min_common
        )
        sims.append((other_user, sim))

    sims.sort(key=lambda x: x[1], reverse=True)
    return sims[:k]

# Ejemplo rápido de prueba: 
k = 10
target_user = 0
knn_users(ui_matrix, target_user, pearson_similarity_users, k, 1)

[(1, 0.0),
 (2, 0.0),
 (3, 0.0),
 (4, 0.0),
 (5, 0.0),
 (6, 0.0),
 (7, 0.0),
 (8, 0.0),
 (9, 0.0),
 (10, 0.0)]

In [15]:
# Usando ahora la similaridad coseno: 
k = 10
target_user = 0
neighbours = knn_users(ui_matrix, target_user, cosine_similarity_users, k, 1)
neighbours

[(48476, 0.7071067811865475),
 (50777, 0.7071067811865475),
 (220909, 0.7071067811865475),
 (287920, 0.7071067811865475),
 (386136, 0.7071067811865475),
 (399520, 0.7071067811865475),
 (476083, 0.7071067811865475),
 (601046, 0.7071067811865475),
 (601166, 0.7071067811865475),
 (696356, 0.7071067811865475)]

### 3) Estimación de las predicciones: 

In [16]:
def deviation_from_mean_prediction(user, item, neighbours):
    """
    Predice el playcount de (user, item) usando deviation from mean.
    neighbours debe ser una lista de tuplas: (neighbour_user, similarity).
    """
    # Obtenemos la media del usuario objetivo sobre sus ítems escuchados:
    user_row = ui_matrix[user]
    if user_row.nnz == 0:
        return 0.0
    user_mean = float(user_row.data.mean())

    num = 0.0
    den = 0.0

    # Iteramos por cada vecino junto con su similaridad:
    for neighbour, sim in neighbours:

        # Si ese vecino no tiene reproducciones sobre esa canción, continuamos al siguiente vecino:
        r_vi = float(ui_matrix[neighbour, item])
        if r_vi <= 0:
            continue
        
        # Calculamos la media del vecino:
        neigh_row = ui_matrix[neighbour]
        if neigh_row.nnz == 0:
            continue
        neigh_mean = float(neigh_row.data.mean())

        # Por un lado, calculamos la desviación de su número de reproducciones de esa canción con respecto
        # a su media y la multiplicamos por su similitud de modo que se dé mayor valor a aquellos vecinos que más
        # se parezcan al usuario objetivo, y todo ello lo dividimos por la suma de todas las similitudes de los vecinos: 
        num += sim * (r_vi - neigh_mean)
        den += abs(sim)

    if den == 0:
        return user_mean

    # Obtenemos y devolvemos resultado final: 
    pred = user_mean + (num / den)
    return max(0.0, float(pred))

# Ejemplo rápido de prueba: 
user = 100
item = 33
deviation_from_mean_prediction(user, item, neighbours)

1.0